In [ ]:
from pathlib import Path
from typing import Union
import logging

import teehr
from teehr.evaluation.spark_session_utils import create_spark_session

logger = logging.getLogger()

In [ ]:
%%time
spark = create_spark_session(
    aws_profile="admin-user",
    start_spark_cluster=True,
    executor_instances=48,
    executor_memory="32g",
    executor_cores=4
)
ev = teehr.RemoteReadWriteEvaluation(
    spark=spark,
    enable_spark_proxy=True
)

In [ ]:
OUTPUT_TABLE_NAME = "configurations_by_location"

In [ ]:
def summarize_primary_locations(
    ev: teehr.Evaluation
) -> None:
    """Summarize primary locations."""
    logger.info(
        f"Summarizing primary locations into a spark dataframe..."
    )
    return ev.spark.sql("""
        SELECT
            location_id as primary_location_id,
            configuration_name,
            variable_name,
            unit_name,
            MIN(reference_time) AS min_reference_time,
            MAX(reference_time) AS max_reference_time,
            MIN(value_time)     AS min_value_time,
            MAX(value_time)     AS max_value_time,
            CAST(NULL AS BIGINT) AS num_members
        FROM iceberg.teehr.primary_timeseries
        GROUP BY primary_location_id, configuration_name, variable_name, unit_name
    """)

def summarize_secondary_locations(
    ev: teehr.Evaluation
) -> None:
    """Summarize secondary locations."""
    logger.info(
        f"Summarizing secondary locations into a spark dataframe..."
    )
    return ev.spark.sql("""
        SELECT
            cf.primary_location_id,
            st.configuration_name,
            st.variable_name,
            st.unit_name,
            MIN(st.reference_time) AS min_reference_time,
            MAX(st.reference_time) AS max_reference_time,
            MIN(st.value_time)     AS min_value_time,
            MAX(st.value_time)     AS max_value_time,
            COUNT(DISTINCT st.member) AS num_members
        FROM iceberg.teehr.secondary_timeseries st
        JOIN iceberg.teehr.location_crosswalks cf
            ON cf.secondary_location_id = st.location_id
        GROUP BY cf.primary_location_id, st.configuration_name, st.variable_name, st.unit_name
    """)

In [ ]:
primary_locations_summary_sdf = summarize_primary_locations(ev=ev)
secondary_locations_summary_sdf = summarize_secondary_locations(ev=ev)

result_sdf = primary_locations_summary_sdf.unionByName(secondary_locations_summary_sdf)

In [ ]:
%%time
ev._write.to_warehouse(
    source_data=result_sdf,
    table_name=OUTPUT_TABLE_NAME,
    write_mode="create_or_replace"
)